In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


15-01 — Load CrisisWeave Final States

In [2]:
# ============================================================
# MODULE 15: RECOMMENDATION
# 15-01: LOAD FINAL CRISISWEAVE STATES
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

# ------------------------------------------------------------
# RESULT DIRECTORIES
# ------------------------------------------------------------

DYNAMIC_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Dynamic_State"
)

LOCATION_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Location_Verification"
)

PRIORITY_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Priority_Location"
)

CONFLICT_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Conflict"
)

FUSION_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Fusion"
)

# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------

DYNAMIC_PATH = os.path.join(
    DYNAMIC_DIR,
    "CrisisWeave_Dynamic_Disaster_State_CW001.json"
)

LOCATION_PATH = os.path.join(
    LOCATION_DIR,
    "CrisisWeave_Location_Verification_CW001.json"
)

PRIORITY_PATH = os.path.join(
    PRIORITY_DIR,
    "CrisisWeave_Next_Priority_Location_CW001.json"
)

CONFLICT_PATH = os.path.join(
    CONFLICT_DIR,
    "CrisisWeave_Conflict_Assessment_CW001_EV001.json"
)

FUSION_PATH = os.path.join(
    FUSION_DIR,
    "CrisisWeave_Fused_State_CW001_EV001.json"
)

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------

def load_json(path):

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

dynamic_state = load_json(
    DYNAMIC_PATH
)

location_verification = load_json(
    LOCATION_PATH
)

priority_location = load_json(
    PRIORITY_PATH
)

conflict_assessment = load_json(
    CONFLICT_PATH
)

fused_state = load_json(
    FUSION_PATH
)

print("=" * 65)
print("RECOMMENDATION INPUTS LOADED")
print("=" * 65)

print("\n✓ Dynamic disaster state")
print("✓ Location verification")
print("✓ Next priority location")
print("✓ Conflict assessment")
print("✓ Conflict-aware fused state")

print("\nAll required Recommendation inputs are available.")

RECOMMENDATION INPUTS LOADED

✓ Dynamic disaster state
✓ Location verification
✓ Next priority location
✓ Conflict assessment
✓ Conflict-aware fused state

All required Recommendation inputs are available.


15-02 — Generate Operational Recommendation

In [3]:
# ============================================================
# 15-02: CRISISWEAVE RECOMMENDATION ENGINE
# ============================================================

# ------------------------------------------------------------
# 1. EXTRACT DYNAMIC STATE
# ------------------------------------------------------------

current_state = dynamic_state.get(
    "current_state",
    {}
)

humanitarian_state = dynamic_state.get(
    "humanitarian_state",
    {}
)

temporal_state = dynamic_state.get(
    "temporal_state",
    {}
)

geographic_state = dynamic_state.get(
    "geographic_state",
    {}
)

response_state = dynamic_state.get(
    "response_state",
    {}
)

evidence_quality = dynamic_state.get(
    "evidence_quality",
    {}
)

# ------------------------------------------------------------
# 2. EXTRACT VALUES
# ------------------------------------------------------------

severity = str(
    current_state.get(
        "severity",
        "UNKNOWN"
    )
).upper()

severity_confidence = current_state.get(
    "state_confidence",
    None
)

operational_risk = str(
    current_state.get(
        "operational_risk",
        "UNKNOWN"
    )
).upper()

humanitarian_category = str(
    humanitarian_state.get(
        "category",
        "UNKNOWN"
    )
)

temporal_trend = str(
    temporal_state.get(
        "trend",
        "UNKNOWN"
    )
).upper()

escalation_state = str(
    temporal_state.get(
        "escalation_state",
        "UNKNOWN"
    )
).upper()

progression_direction = str(
    temporal_state.get(
        "progression_direction",
        "UNKNOWN"
    )
).upper()

verification_status = str(
    location_verification.get(
        "verification_status",
        location_verification.get(
            "status",
            "UNKNOWN"
        )
    )
).upper()

verification_score = location_verification.get(
    "verification_score",
    None
)

conflict_status = str(
    evidence_quality.get(
        "conflict_status",
        conflict_assessment.get(
            "overall_conflict",
            False
        )
    )
).upper()

overall_conflict = evidence_quality.get(
    "overall_conflict",
    False
)

response_status = str(
    response_state.get(
        "status",
        "UNKNOWN"
    )
).upper()

# ------------------------------------------------------------
# 3. PRIORITY LOCATION
# ------------------------------------------------------------

selected_priority = priority_location.get(
    "selected_location",
    priority_location.get(
        "next_priority_location",
        {}
    )
)

if not isinstance(
    selected_priority,
    dict
):
    selected_priority = {}

priority_name = selected_priority.get(
    "name",
    "Unknown"
)

priority_type = selected_priority.get(
    "type",
    "Unknown"
)

priority_lat = selected_priority.get(
    "latitude",
    None
)

priority_lon = selected_priority.get(
    "longitude",
    None
)

priority_score = selected_priority.get(
    "priority_score",
    None
)

# ------------------------------------------------------------
# 4. FALLBACK:
# IF SAVED JSON STRUCTURE IS DIFFERENT,
# SEARCH COMMON FIELDS
# ------------------------------------------------------------

if priority_name == "Unknown":

    for key in [
        "name",
        "selected_name",
        "priority_name"
    ]:

        if key in priority_location:

            priority_name = priority_location[key]
            break

if priority_score is None:

    for key in [
        "priority_score",
        "selected_priority_score",
        "score"
    ]:

        if key in priority_location:

            priority_score = priority_location[key]
            break

# ------------------------------------------------------------
# 5. DETERMINE VERIFICATION
# ------------------------------------------------------------

verification_valid = (
    verification_status
    in [
        "VERIFIED",
        "PARTIALLY VERIFIED"
    ]
)

# ------------------------------------------------------------
# 6. DETERMINE HIGH-SEVERITY CONDITION
# ------------------------------------------------------------

high_severity = (
    severity == "HIGH"
)

# ------------------------------------------------------------
# 7. DETERMINE ESCALATION
# ------------------------------------------------------------

increasing_trend = (
    temporal_trend
    in [
        "INCREASING",
        "ESCALATING",
        "RISING"
    ]
)

# ------------------------------------------------------------
# 8. DETERMINE PRIORITY ELIGIBILITY
# ------------------------------------------------------------

priority_available = (
    priority_name != "Unknown"
    and priority_lat is not None
    and priority_lon is not None
)

# ------------------------------------------------------------
# 9. PRIMARY RECOMMENDATION
# ------------------------------------------------------------

if (
    high_severity
    and verification_valid
    and increasing_trend
):

    recommendation_level = (
        "IMMEDIATE RESPONSE"
    )

    primary_recommendation = (
        "Immediate disaster response is recommended. "
        "The incident is assessed as HIGH severity, "
        "the location is geographically verified or "
        "partially verified, and the temporal evidence "
        "indicates an increasing trend."
    )

elif (
    high_severity
    and verification_valid
):

    recommendation_level = (
        "HIGH PRIORITY RESPONSE"
    )

    primary_recommendation = (
        "High-priority disaster response is recommended. "
        "The incident is assessed as HIGH severity and "
        "the reported location is sufficiently verified."
    )

elif high_severity:

    recommendation_level = (
        "URGENT VALIDATION"
    )

    primary_recommendation = (
        "Urgent validation is recommended before "
        "full operational escalation because the "
        "incident is assessed as HIGH severity but "
        "location verification is insufficient."
    )

elif severity == "MODERATE":

    recommendation_level = (
        "MONITOR AND VERIFY"
    )

    primary_recommendation = (
        "Continue monitoring and obtain additional "
        "evidence before escalating the response."
    )

else:

    recommendation_level = (
        "CONTINUE MONITORING"
    )

    primary_recommendation = (
        "Continue monitoring the incident and "
        "reassess when new evidence becomes available."
    )

# ------------------------------------------------------------
# 10. PRIORITY LOCATION ACTION
# ------------------------------------------------------------

if (
    high_severity
    and verification_valid
    and priority_available
):

    priority_action = (
        f"Prioritize {priority_name} "
        f"({priority_type}) as the next location "
        f"for assessment or preparedness action."
    )

else:

    priority_action = (
        "No next-location priority action is "
        "recommended until the required eligibility "
        "conditions are satisfied."
    )

# ------------------------------------------------------------
# 11. TEMPORAL ACTION
# ------------------------------------------------------------

if increasing_trend:

    temporal_action = (
        f"Maintain active monitoring in the "
        f"{progression_direction} progression direction "
        f"because the disaster state is showing an "
        f"increasing trend."
    )

else:

    temporal_action = (
        "Continue temporal monitoring for changes "
        "in severity and progression direction."
    )

# ------------------------------------------------------------
# 12. HUMANITARIAN ACTION
# ------------------------------------------------------------

humanitarian_action = (
    f"Humanitarian response planning should consider "
    f"the reported information category: "
    f"{humanitarian_category}."
)

# ------------------------------------------------------------
# 13. RESPONSE STATUS ACTION
# ------------------------------------------------------------

if response_status in [
    "NOT_STARTED",
    "PENDING",
    "UNKNOWN"
]:

    response_action = (
        "Response coordination should be initiated "
        "or verified against the latest available "
        "response information."
    )

else:

    response_action = (
        f"Continue monitoring the response state "
        f"({response_status}) and update it as new "
        f"response events become available."
    )

# ------------------------------------------------------------
# 14. CONFLICT ACTION
# ------------------------------------------------------------

if overall_conflict is True:

    conflict_action = (
        "Because conflicting evidence is present, "
        "additional validation should be obtained "
        "before making irreversible operational decisions."
    )

else:

    conflict_action = (
        "No major evidence conflict is currently "
        "identified; the fused state can be used "
        "for decision support."
    )

# ------------------------------------------------------------
# 15. OVERALL DECISION
# ------------------------------------------------------------

if (
    recommendation_level
    == "IMMEDIATE RESPONSE"
):

    overall_decision = (
        "ACT NOW AND PRIORITIZE NEXT LOCATION"
    )

elif (
    recommendation_level
    == "HIGH PRIORITY RESPONSE"
):

    overall_decision = (
        "PRIORITIZE RESPONSE"
    )

elif (
    recommendation_level
    == "URGENT VALIDATION"
):

    overall_decision = (
        "VALIDATE BEFORE ESCALATION"
    )

else:

    overall_decision = (
        "MONITOR AND REASSESS"
    )

# ------------------------------------------------------------
# 16. BUILD RECOMMENDATION OBJECT
# ------------------------------------------------------------

recommendation = {

    "incident_id": "CW-001",

    "recommendation_level":
        recommendation_level,

    "overall_decision":
        overall_decision,

    "primary_recommendation":
        primary_recommendation,

    "current_disaster_state": {

        "severity":
            severity,

        "operational_risk":
            operational_risk,

        "state_confidence":
            severity_confidence

    },

    "location_verification": {

        "status":
            verification_status,

        "verification_score":
            verification_score

    },

    "temporal_assessment": {

        "trend":
            temporal_trend,

        "escalation_state":
            escalation_state,

        "progression_direction":
            progression_direction

    },

    "humanitarian_assessment": {

        "category":
            humanitarian_category

    },

    "evidence_quality": {

        "conflict_status":
            conflict_status,

        "overall_conflict":
            overall_conflict

    },

    "next_priority_location": {

        "name":
            priority_name,

        "type":
            priority_type,

        "latitude":
            priority_lat,

        "longitude":
            priority_lon,

        "priority_score":
            priority_score

    },

    "recommended_actions": [

        primary_recommendation,

        priority_action,

        temporal_action,

        humanitarian_action,

        response_action,

        conflict_action

    ]

}

# ------------------------------------------------------------
# 17. DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CRISISWEAVE RECOMMENDATION")
print("=" * 70)

print(
    "\nRecommendation Level:"
)
print(
    "  ",
    recommendation_level
)

print(
    "\nOverall Decision:"
)
print(
    "  ",
    overall_decision
)

print(
    "\nPrimary Recommendation:"
)
print(
    "  ",
    primary_recommendation
)

print(
    "\nNext Priority Location:"
)
print(
    "  ",
    priority_name
)

print(
    "  Type:",
    priority_type
)

print(
    "  Score:",
    priority_score
)

print(
    "\nTemporal Direction:"
)
print(
    "  ",
    progression_direction
)

print(
    "\nTemporal Trend:"
)
print(
    "  ",
    temporal_trend
)

print(
    "\nHumanitarian Category:"
)
print(
    "  ",
    humanitarian_category
)

print(
    "\nConflict:"
)
print(
    "  ",
    overall_conflict
)

print(
    "\n" + "=" * 70
)


CRISISWEAVE RECOMMENDATION

Recommendation Level:
   IMMEDIATE RESPONSE

Overall Decision:
   ACT NOW AND PRIORITIZE NEXT LOCATION

Primary Recommendation:
   Immediate disaster response is recommended. The incident is assessed as HIGH severity, the location is geographically verified or partially verified, and the temporal evidence indicates an increasing trend.

Next Priority Location:
   AMJ Hospital
  Type: Hospital
  Score: 0.8

Temporal Direction:
   NE

Temporal Trend:
   INCREASING

Humanitarian Category:
   requests_or_urgent_needs

Conflict:
   False



15-03 — Save Recommendation

In [4]:
# ============================================================
# 15-03: SAVE CRISISWEAVE RECOMMENDATION
# ============================================================

RECOMMENDATION_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Recommendation"
)

os.makedirs(
    RECOMMENDATION_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# JSON OUTPUT
# ------------------------------------------------------------

recommendation_json_path = os.path.join(
    RECOMMENDATION_DIR,
    "CrisisWeave_Recommendation_CW001.json"
)

with open(
    recommendation_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        recommendation,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# CSV SUMMARY
# ------------------------------------------------------------

recommendation_summary = {

    "incident_id":
        recommendation["incident_id"],

    "recommendation_level":
        recommendation["recommendation_level"],

    "overall_decision":
        recommendation["overall_decision"],

    "severity":
        recommendation[
            "current_disaster_state"
        ]["severity"],

    "operational_risk":
        recommendation[
            "current_disaster_state"
        ]["operational_risk"],

    "location_verification":
        recommendation[
            "location_verification"
        ]["status"],

    "temporal_trend":
        recommendation[
            "temporal_assessment"
        ]["trend"],

    "progression_direction":
        recommendation[
            "temporal_assessment"
        ]["progression_direction"],

    "humanitarian_category":
        recommendation[
            "humanitarian_assessment"
        ]["category"],

    "conflict_status":
        recommendation[
            "evidence_quality"
        ]["conflict_status"],

    "next_priority_location":
        recommendation[
            "next_priority_location"
        ]["name"],

    "next_priority_type":
        recommendation[
            "next_priority_location"
        ]["type"],

    "priority_score":
        recommendation[
            "next_priority_location"
        ]["priority_score"]

}

recommendation_csv_path = os.path.join(
    RECOMMENDATION_DIR,
    "CrisisWeave_Recommendation_CW001.csv"
)

pd.DataFrame(
    [recommendation_summary]
).to_csv(
    recommendation_csv_path,
    index=False
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODULE 15 — RECOMMENDATION COMPLETED")
print("=" * 70)

print("\nJSON saved:")
print(
    recommendation_json_path
)

print("\nCSV saved:")
print(
    recommendation_csv_path
)

print("\nFinal decision:")
print(
    recommendation["overall_decision"]
)

print("\n" + "=" * 70)


MODULE 15 — RECOMMENDATION COMPLETED

JSON saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Recommendation/CrisisWeave_Recommendation_CW001.json

CSV saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Recommendation/CrisisWeave_Recommendation_CW001.csv

Final decision:
ACT NOW AND PRIORITIZE NEXT LOCATION

